# Email Nickname Extractor

Extract informal nicknames of financial advisors from corporate emails using an LLM pipeline.

**Pipeline:** Generate synthetic emails → Extract nicknames with LLM → Evaluate precision/recall

## 0. Setup

In [ ]:
!pip install openai -q

import json, random, uuid, os, sys
from datetime import datetime, timedelta
from pathlib import Path
from openai import OpenAI

# Set your API key
os.environ['DEEPSEEK_API_KEY'] = 'YOUR_KEY_HERE'

client = OpenAI(
    api_key=os.environ['DEEPSEEK_API_KEY'],
    base_url='https://api.deepseek.com'
)

Path('data').mkdir(exist_ok=True)
print('Setup complete.')

## 1. Define Advisor Profiles

Each advisor has a formal name and optionally one or more nicknames. The nicknames are our ground truth.

3 out of 10 advisors have no nicknames. This tests whether the extractor hallucinates nicknames for people who don't have any.

In [ ]:
ADVISORS = [
    {'id': 'adv_001', 'formal_name': 'Robert Chen', 'email': 'robert.chen@capitaledge.com',
     'nicknames': ['Bobby', 'Bob'], 'role': 'Senior Portfolio Manager'},
    {'id': 'adv_002', 'formal_name': 'Elizabeth Warren-Hughes', 'email': 'e.warrenhughes@capitaledge.com',
     'nicknames': ['Liz', 'Lizzie'], 'role': 'Head of Fixed Income'},
    {'id': 'adv_003', 'formal_name': 'James Okonkwo', 'email': 'james.okonkwo@capitaledge.com',
     'nicknames': [], 'role': 'Quantitative Analyst'},
    {'id': 'adv_004', 'formal_name': 'Katherine M\u00fcller', 'email': 'k.muller@capitaledge.com',
     'nicknames': ['Kat'], 'role': 'Risk Manager'},
    {'id': 'adv_005', 'formal_name': 'Alexander Petrov', 'email': 'a.petrov@capitaledge.com',
     'nicknames': ['Alex', 'Sasha'], 'role': 'Derivatives Trader'},
    {'id': 'adv_006', 'formal_name': 'Priya Sharma', 'email': 'priya.sharma@capitaledge.com',
     'nicknames': [], 'role': 'Compliance Officer'},
    {'id': 'adv_007', 'formal_name': 'William Thornton III', 'email': 'w.thornton@capitaledge.com',
     'nicknames': ['Will', 'Billy T'], 'role': 'Client Relations Director'},
    {'id': 'adv_008', 'formal_name': 'Margaret Liu', 'email': 'margaret.liu@capitaledge.com',
     'nicknames': ['Maggie', 'Mags'], 'role': 'ESG Research Lead'},
    {'id': 'adv_009', 'formal_name': 'Christopher Delacroix', 'email': 'c.delacroix@capitaledge.com',
     'nicknames': ['Chris'], 'role': 'Head of Equity Research'},
    {'id': 'adv_010', 'formal_name': 'Tomoko Hayashi', 'email': 't.hayashi@capitaledge.com',
     'nicknames': [], 'role': 'Asia Pacific Strategist'},
]

print(f'Defined {len(ADVISORS)} advisors')
print(f'  With nicknames: {sum(1 for a in ADVISORS if a["nicknames"])}')
print(f'  Without nicknames: {sum(1 for a in ADVISORS if not a["nicknames"])}')

## 2. Email Templates

Nicknames appear in varied positions: greetings, mid-sentence, sign-offs. This variety makes extraction harder and more realistic.

In [ ]:
TEMPLATES_WITH_NICKNAME = [
    {'subject': 'Q{quarter} Portfolio Rebalance',
     'body': 'Hey {nickname},\n\nJust finished the Q{quarter} rebalance analysis. We\'re overweight in tech by about 3.2% relative to benchmark. Recommending we trim NVDA and rotate into healthcare names.\n\nCan you take a look at the proposed allocations before EOD?\n\nCheers,\n{sender_first}'},
    {'subject': 'Re: Client meeting prep',
     'body': 'Thanks for the deck. One thing \u2014 {nickname}, can you double check the Sharpe ratio calculations on slide 7? The numbers look off compared to what Bloomberg is showing.\n\nAlso, the Hendersons want to discuss their charitable trust allocation, so let\'s prep some tax-efficient options.\n\n{sender_first}'},
    {'subject': 'Lunch?',
     'body': '{nickname}! Free for lunch today? The new place on Threadneedle St is supposed to be great. We can talk about the emerging markets thesis over food.\n\nLet me know,\n{sender_first}'},
    {'subject': 'FYI: Macro outlook changed',
     'body': 'Heads up team,\n\nThe Fed minutes just dropped and it\'s more hawkish than expected. {nickname}, this directly impacts your duration positioning \u2014 might want to revisit the 10Y allocation.\n\nI\'ve attached the summary. Let\'s regroup at 3pm.\n\nBest,\n{sender_first}'},
    {'subject': 'Re: Re: Risk limits breach',
     'body': 'Understood. {nickname}, I\'ve already flagged this with compliance. The VaR breach was transient \u2014 driven by the options expiry. We\'re back within limits as of this morning.\n\nHappy to walk through the P&L attribution if needed.\n\nThanks,\n{sender_first}'},
    {'subject': 'Great call today',
     'body': 'Nice work on the Meridian pitch today, {nickname}. The client was clearly impressed with the factor analysis. I think we\'re in a strong position for the mandate.\n\nLet\'s debrief tomorrow morning.\n\nBest,\n{sender_first}'},
    {'subject': 'Quick favor',
     'body': 'Hey {nickname}, could you pull the historical drawdown data for our EM sovereign book? Need it for the board deck by Thursday.\n\nAppreciate it!\n{sender_first}'},
    {'subject': 'Re: New hire onboarding',
     'body': 'Good idea. {nickname}, since you went through onboarding most recently, would you mind being the buddy for the new analyst? They start on the 15th and will be sitting on your floor.\n\nThanks,\n{sender_first}'},
    {'subject': 'Weekend reading',
     'body': '{nickname} \u2014 thought you\'d find this interesting. Bridgewater just published their updated macro framework. Particularly relevant given your view on the yield curve inversion.\n\nLink attached.\n\nEnjoy,\n{sender_first}'},
    {'subject': 'Urgent: Position sizing',
     'body': 'We need to cut the Japan exposure by half before Tokyo open. {nickname}, you have the authority on this book \u2014 can you action this ASAP? I\'ll explain on the call in 10 min.\n\n{sender_first}'},
]

TEMPLATES_FORMAL = [
    {'subject': 'Monthly performance report',
     'body': 'Hi {formal_first},\n\nPlease find attached the monthly performance attribution for your book. AUM is up 2.3% net of fees, outperforming the benchmark by 45bps.\n\nLet me know if you have questions.\n\nRegards,\n{sender_first}'},
    {'subject': 'Compliance training reminder',
     'body': 'Dear {formal_first},\n\nThis is a reminder that the annual compliance training module is due by end of month. Please complete it at your earliest convenience.\n\nThank you,\n{sender_first}'},
    {'subject': 'Re: Data request',
     'body': '{formal_first},\n\nThe dataset you requested is now available in the shared drive. It covers all equity trades from 2020 to present. Let me know if you need the fixed income data as well.\n\nBest,\n{sender_first}'},
]

print(f'Defined {len(TEMPLATES_WITH_NICKNAME)} nickname templates + {len(TEMPLATES_FORMAL)} formal templates')

## 3. Generate Synthetic Emails

120 emails between the 10 advisors. When a recipient has nicknames, we use one 70% of the time. The other 30% they get called by their real first name, because in real life people don't always use nicknames.

In [ ]:
random.seed(42)
emails = []
base_date = datetime(2025, 6, 1)

for i in range(120):
    sender, recipient = random.sample(ADVISORS, 2)
    sender_first = sender['formal_name'].split()[0]
    recipient_first = recipient['formal_name'].split()[0]
    use_nickname = len(recipient['nicknames']) > 0 and random.random() < 0.7

    if use_nickname:
        nickname = random.choice(recipient['nicknames'])
        template = random.choice(TEMPLATES_WITH_NICKNAME)
    else:
        nickname = recipient_first
        template = random.choice(TEMPLATES_FORMAL + TEMPLATES_WITH_NICKNAME)

    body = template['body'].format(nickname=nickname, formal_first=recipient_first,
                                   sender_first=sender_first, quarter=random.randint(1, 4))
    subject = template['subject'].format(quarter=random.randint(1, 4))
    timestamp = (base_date + timedelta(days=random.randint(0, 180),
                 hours=random.randint(7, 19), minutes=random.randint(0, 59))).isoformat()

    emails.append({'id': str(uuid.uuid4()), 'timestamp': timestamp,
                   'from': {'name': sender['formal_name'], 'email': sender['email']},
                   'to': {'name': recipient['formal_name'], 'email': recipient['email']},
                   'subject': subject, 'body': body})

with open('data/emails.json', 'w') as f:
    json.dump(emails, f, indent=2)

ground_truth = {a['id']: {'formal_name': a['formal_name'], 'email': a['email'],
                'known_nicknames': a['nicknames']} for a in ADVISORS}
with open('data/ground_truth.json', 'w') as f:
    json.dump(ground_truth, f, indent=2)

nickname_count = sum(1 for e in emails for a in ADVISORS
                     if a['email'] == e['to']['email']
                     and any(n in e['body'] for n in a['nicknames']))

print(f'Generated {len(emails)} emails')
print(f'Emails containing a real nickname: {nickname_count}/{len(emails)}')
print(f'\n--- Sample email ---')
print(f'From: {emails[1]["from"]["name"]}')
print(f'To:   {emails[1]["to"]["name"]}')
print(f'Subj: {emails[1]["subject"]}\n')
print(emails[1]['body'])

## 4. Nickname Extraction

For each advisor, we collect all their emails (sent and received) and send them to the LLM in a single prompt. Per-person batching gives the model enough context to cross-reference nickname usage across multiple emails.

We use DeepSeek via the OpenAI-compatible API. Swapping to any other provider (Claude, GPT-4, Gemini) is a one-line change to `base_url`.

In [ ]:
def get_emails_for_person(email_address):
    return [e for e in emails
            if e['from']['email'] == email_address or e['to']['email'] == email_address]

def extract_nicknames(formal_name, email_address, person_emails):
    email_block = ''
    for i, e in enumerate(person_emails, 1):
        direction = 'SENT' if e['from']['email'] == email_address else 'RECEIVED'
        other = e['to']['name'] if direction == 'SENT' else e['from']['name']
        email_block += f'\n--- Email {i} ({direction}, other party: {other}) ---\n'
        email_block += f'Subject: {e["subject"]}\n{e["body"]}\n'

    prompt = f"""You are analyzing corporate emails to identify nicknames used for a specific person.

TARGET PERSON:
  Formal name: {formal_name}
  Email: {email_address}

Below are {len(person_emails)} emails this person sent or received. Your task:
1. Identify any NICKNAMES other people use to refer to {formal_name} in received emails
2. Also identify any nicknames {formal_name} uses to sign their OWN sent emails

A nickname is an informal or shortened version of someone's name that differs
from their standard first name. For example, "Bobby" for "Robert" or "Liz" for
"Elizabeth" would be nicknames, but "Robert" for "Robert Chen" is NOT a nickname.

IMPORTANT:
- Only report names clearly used AS nicknames for this specific person
- Do NOT report their standard first name as a nickname
- If no nicknames are found, return an empty list
- Be conservative: only include names you are confident about

EMAILS:
{email_block}

Respond with ONLY a JSON object in this exact format, no other text:
{{{{
  "nicknames": ["nickname1", "nickname2"],
  "evidence": [
    {{{{"nickname": "nickname1", "example": "brief quote showing usage", "confidence": "high/medium/low"}}}}
  ]
}}}}

If no nicknames found, respond with:
{{{{"nicknames": [], "evidence": []}}}}"""

    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1024
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1].rsplit('```', 1)[0].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'  WARNING: Could not parse response for {formal_name}')
        return {'nicknames': [], 'evidence': [], 'parse_error': True}

print('Extraction function ready.')

In [ ]:
people = {}
for e in emails:
    people[e['from']['email']] = e['from']['name']
    people[e['to']['email']] = e['to']['name']

print(f'Extracting nicknames for {len(people)} people...\n')

results = {}
for email_address, formal_name in sorted(people.items()):
    person_emails = get_emails_for_person(email_address)
    print(f'  {formal_name} ({len(person_emails)} emails)...', end=' ')
    extraction = extract_nicknames(formal_name, email_address, person_emails)
    results[email_address] = {
        'formal_name': formal_name, 'num_emails_analyzed': len(person_emails),
        'extracted_nicknames': extraction.get('nicknames', []),
        'evidence': extraction.get('evidence', []),
    }
    nicks = extraction.get('nicknames', [])
    print(f'Found: {nicks}' if nicks else 'No nicknames')

with open('data/extracted_nicknames.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nDone! Results saved.')

## 5. Evaluation

Compare extracted nicknames against ground truth. In a financial context, precision matters more than recall: calling a client by a wrong nickname is worse than not knowing their nickname at all.

In [ ]:
def normalize(name):
    return name.strip().lower()

gt = json.load(open('data/ground_truth.json'))
ext = json.load(open('data/extracted_nicknames.json'))

gt_by_email = {info['email']: set(normalize(n) for n in info['known_nicknames'])
               for info in gt.values()}

total_tp, total_fp, total_fn = 0, 0, 0

print('=' * 55)
print('NICKNAME EXTRACTION EVALUATION')
print('=' * 55)

for email_addr, ext_info in sorted(ext.items()):
    known = gt_by_email.get(email_addr, set())
    found = set(normalize(n) for n in ext_info['extracted_nicknames'])
    tp = known & found
    fp = found - known
    fn = known - found
    total_tp += len(tp); total_fp += len(fp); total_fn += len(fn)

    status = 'PASS' if not fp and not fn else 'FAIL'
    name = ext_info['formal_name']
    print(f'\n  [{status}] {name}')
    print(f'    Known:     {sorted(known) if known else "(none)"}')
    print(f'    Extracted: {sorted(found) if found else "(none)"}')
    if fp: print(f'    FALSE POS: {sorted(fp)}')
    if fn: print(f'    MISSED:    {sorted(fn)}')

prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
rec  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

print(f'\n{"=" * 55}')
print(f'  Precision: {prec:.0%}')
print(f'  Recall:    {rec:.0%}')
print(f'  F1 Score:  {f1:.0%}')
print(f'  TP: {total_tp}  FP: {total_fp}  FN: {total_fn}')
print(f'{"=" * 55}')